In [4]:
!pip install pyarrow

# Browse Data in AUTokens and find AU EN characteristics

In [5]:
import pandas as pd
df = pd.read_parquet("~/data/AUTokens50/part_48.parquet")
df.head()

,text,id,dump,url,date,file_path,language,language_score,token_count
1939,"FOR DAVID GULPILIL, SUNSMART AND THE EARTH, WI...",<urn:uuid:7c02bd39-1918-4f60-b99e-0eece2c8fdaa>,CC-MAIN-2024-22,https://sue.coulstock.id.au/the-kingfisher-the...,2024-05-18T01:07:03Z,s3://commoncrawl/crawl-data/CC-MAIN-2024-22/se...,en,0.979648,4908
3920,"MAC News 4 2023\nDear MAC Community,\nAs the t...",<urn:uuid:1f1aa282-8422-4a71-8c1d-e43f1c1cf50f>,CC-MAIN-2024-22,http://www.mountalexandercollege.vic.edu.au/ab...,2024-05-19T01:32:26Z,s3://commoncrawl/crawl-data/CC-MAIN-2024-22/se...,en,0.962712,4640
4284,Kloe & Alex\n“I highly recommend doing a birth...,<urn:uuid:df8451d9-f92b-45f0-9d51-f3a52b6ecc12>,CC-MAIN-2024-22,https://calmbirth.com.au/the-calmbirth-of-baby...,2024-05-19T02:51:59Z,s3://commoncrawl/crawl-data/CC-MAIN-2024-22/se...,en,0.986469,1504
5160,I want to talk to you about what most people t...,<urn:uuid:4c5bbfae-c171-401e-97f6-0577198731f2>,CC-MAIN-2024-22,https://mould.net.au/blogs/news/can-mould-caus...,2024-05-19T02:55:26Z,s3://commoncrawl/crawl-data/CC-MAIN-2024-22/se...,en,0.969435,5955
6006,The truly independent voice of Australian moto...,<urn:uuid:73a79010-479c-4981-94a1-19aed552a37e>,CC-MAIN-2024-22,https://www.anyauto.com.au/about-anyauto/,2024-05-19T01:08:25Z,s3://commoncrawl/crawl-data/CC-MAIN-2024-22/se...,en,0.974305,1937


### random pick a line in df

In [6]:
row = df.sample(n=1).iloc[0]

row_index = row.name
row_text = row["text"]

print("row_index:", row_index)
print(row_text)

row_index: 359288
HELP! MY CHILD HAS BEEN DIAGNOSED WITH COELIAC DISEASE
First of all, you are not alone! My eldest daughter Alyssia has Coeliac Disease. As a family we have experienced massive wins and continue to pick ourselves up from the low points.
I have been wanting to get my initial experience down in writing for months now, if I help 1 other parent of a Coeliac child prepare for what is ahead then I will be delighted.
Here are my thoughts and suggestions to help you through the first 18 months post diagnosis.
Educate yourself and be prepared to educate others
People will say ‘It’s an intolerance, gluten is natural it won’t harm’.
Wow, I wish! There is no “A little bit Coeliac” You either have Coeliac or you do not. It only takes a crumb of gluten to trigger the autoimmune reaction in the small intestine of someone with coeliac disease and people wont understand this unless you explain it to them.
This goes for daycare centres and School to, don’t expect your teacher to know an

### content details and analysis

In [10]:
df["text"][805916]

805916    During these retreats, you will have the oppor...
805916    According to a study published in the journal ...
Name: text, dtype: str

### Why This Text is Not Australian English

The medical text about cannabis and cardiovascular disease in the output above is **not Australian English**. Here's why:

1. **Geographic References**: The text explicitly mentions "Canadian adults" and the "U.S. Centers for Disease Control and Prevention", anchoring the content in North American context.

2. **Healthcare System Terms**: References to "Medicaid" and "Obamacare" are exclusively American healthcare programs. Australia has a completely different system (Medicare).

3. **Content Focus**: The discussion centers on North American health risks and statistics, not Australian health concerns.

### How Different English Variants Use Different Terminology

English-speaking countries have distinct vocabulary in several key domains:

#### 1. **Tax Systems**
- **USA**: IRS, SSN (Social Security Number), W-2 form, 1099 form, EIN (Employer Identification Number)
- **UK**: HMRC, NI number (National Insurance Number), PAYE (Pay As You Earn), VAT (Value Added Tax)
- **Australia**: ATO (Australian Taxation Office), TFN (Tax File Number), ABN (Australian Business Number)

#### 2. **Education System**
- **USA**:SAT, high school diploma
- **UK**: GCSE, A-Level
- **Australia**: HSC (Higher School Certificate), VCE (Victorian Certificate of Education), ATAR (Australian Tertiary Admission Rank)

#### 3. **Legal & Administrative**
- **USA**: DMV (Department of Motor Vehicles), SSN
- **UK**: DVLA (Driver and Vehicle Licensing Agency), MOT test, Council tax, HOA (Homeowners Association)
- **Australia**: RTA (Road and Maritime Services in NSW), Council rates, local government areas

#### 4. **Healthcare**
- **USA**: Medicaid, Medicare (federal program), HIPAA
- **UK**: NHS (National Health Service)
- **Australia**: Medicare (universal healthcare), PBS (Pharmaceutical Benefits Scheme), bulk billing

### Detection Strategy

By identifying these **signal words** from specific domains (tax, education, legal, healthcare), we can automatically classify text origins. If a text contains "IRS" or "Medicaid", it's likely American. If it mentions "HMRC" or "GCSE", it's likely British. If it references "ATO" or "ATAR", it's Australian English.

### Risks
Some Acronym have multiple meanings, they might both be used in daily life and in specific domains, they should not be directly used as signal words.
Besides, if a context contains specific domain words from both AU and other countries, it might be an article written by Australian comparing industries in different countries, these samples should not be removed.

## Using signal words to find out non-australian english and topics for au en contents

The following function uses signal words in specific domain to find out and remove a few non-au samples.

In [15]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score


# 1) non-au signal words
NON_AU = [
    "irs", "ssn", "social security number", "ein", "w-2", "1099",
    "medicaid", "obamacare", "dmv", "gpa", "hoa", "sales tax",
    "hmrc", "national insurance number", "ni number", "nhs",
    "gcse", "a-level", "dvla", "mot test", "council tax", "paye", "vat"
]


def contains_non_au_signal(text: str) -> bool:
    text = str(text).lower()
    for kw in NON_AU:
        if re.search(r"\b" + re.escape(kw) + r"\b", text):
            return True
    return False


# 2) skip non-au contents
def filter_non_au(df, text_col="text"):
    df = df.copy()
    df["has_non_au_signal"] = df[text_col].apply(contains_non_au_signal)
    kept = df[~df["has_non_au_signal"]].copy()
    removed = df[df["has_non_au_signal"]].copy()
    return kept, removed


# 3) cluster topics for au contents
def choose_best_k(X, k_range=range(5, 16), random_state=42):
    best_k = None
    best_score = -1

    # pick samples
    n_samples = X.shape[0]
    sample_size = min(5000, n_samples)

    for k in k_range:
        model = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            batch_size=1024,
            n_init="auto"
        )
        labels = model.fit_predict(X)

        # silhouette
        if len(set(labels)) < 2:
            continue

        if n_samples > sample_size:
            idx = np.random.RandomState(random_state).choice(
                n_samples, sample_size, replace=False
            )
            score = silhouette_score(X[idx], labels[idx])
        else:
            score = silhouette_score(X, labels)

        if score > best_score:
            best_score = score
            best_k = k

    return best_k, best_score


# 4) add topic label for clusters
def get_cluster_keywords(tfidf, model, top_n=5):
    terms = np.array(tfidf.get_feature_names_out())
    cluster_keywords = {}

    for i, center in enumerate(model.cluster_centers_):
        top_idx = center.argsort()[::-1][:top_n]
        cluster_keywords[i] = ", ".join(terms[top_idx])

    return cluster_keywords


# 5) main function
def cluster_topics_after_filtering(
    df,
    text_col="text",
    min_df=5,
    max_df=0.5,
    max_features=20000,
    k_range=range(5, 16),
    null_quantile=0.9,
    random_state=42
):
    df = df.copy()

    # Step A: remove non-au texts
    kept, removed = filter_non_au(df, text_col=text_col)

    # return if too little samples
    if len(kept) < 10:
        kept["cluster_id"] = None
        kept["topic"] = None
        kept["topic_reason"] = "Not enough remaining documents after filtering"
        return kept, removed, None

    # Step B: TF-IDF
    tfidf = TfidfVectorizer(
        stop_words="english",
        lowercase=True,
        min_df=min_df,
        max_df=max_df,
        ngram_range=(1, 2),
        max_features=max_features
    )
    X = tfidf.fit_transform(kept[text_col].fillna("").astype(str))

    # Step C: choose k
    best_k, best_score = choose_best_k(X, k_range=k_range, random_state=random_state)
    if best_k is None:
        kept["cluster_id"] = None
        kept["topic"] = None
        kept["topic_reason"] = "Could not determine a stable clustering structure"
        return kept, removed, None

    # Step D: clustering
    cluster_model = MiniBatchKMeans(
        n_clusters=best_k,
        random_state=random_state,
        batch_size=1024,
        n_init="auto"
    )
    cluster_ids = cluster_model.fit_predict(X)

    kept["cluster_id"] = cluster_ids

    # Step E: cluster topic label
    cluster_keywords = get_cluster_keywords(tfidf, cluster_model, top_n=5)

    # Step F: calculate distance
    distances = cluster_model.transform(X)
    min_dist = distances.min(axis=1)

    # too faraway => topic = null
    dist_threshold = np.quantile(min_dist, null_quantile)

    topics = []
    reasons = []

    for cid, dist in zip(cluster_ids, min_dist):
        if dist > dist_threshold:
            topics.append(None)
            reasons.append("Too far from cluster centre")
        else:
            topics.append(cluster_keywords[cid])
            reasons.append(f"Assigned to cluster {cid}")

    kept["topic"] = topics
    kept["topic_reason"] = reasons
    kept["cluster_distance"] = min_dist

    metadata = {
        "best_k": best_k,
        "silhouette_score": best_score,
        "distance_threshold": float(dist_threshold),
        "cluster_keywords": cluster_keywords
    }

    return kept, removed, metadata

In [16]:
# Step 1: ramdomly pick 20000 samples
df_sample = df.sample(n=min(20000, len(df)), random_state=42).reset_index(drop=True)

# Step 2: remove non-au lines using signal words in fields like tax
df_sample = df_sample[df_sample["text"].notna()]
df_sample = df_sample[df_sample["text"].str.len() > 50]
df_sample["text"] = df_sample["text"].str.slice(0, 1000)

# Step 3: clustering
kept_df, removed_df, meta = cluster_topics_after_filtering(df_sample)

# Step 4: results
print(meta)
print(kept_df["topic"].value_counts(dropna=False))

{'best_k': 11, 'silhouette_score': 0.00326447778823304, 'distance_threshold': 0.9969016513711304, 'cluster_keywords': {0: 'day, race, racing, wedding, time', 1: 'game, season, players, games, club', 2: 'cleaning, service, services, moving, furniture', 3: 'women, god, men, female, gender', 4: 'government, minister, australia, federal, australian', 5: 'time, like, people, just, life', 6: 'business, businesses, marketing, small, small business', 7: 'property, market, loan, home, financial', 8: 'australia, year, new, years, australian', 9: 'car, war, cars, vehicle, vehicles', 10: 'ebook, removalists, copyright, site, treasure'}}
topic
time, like, people, just, life                            5983
australia, year, new, years, australian                   5147
NaN                                                       1998
government, minister, australia, federal, australian      1814
property, market, loan, home, financial                   1129
game, season, players, games, club            

In [17]:
print("Total:", len(df_sample))
print("Kept:", len(kept_df))
print("Removed:", len(removed_df))

Total: 20000
Kept: 19981
Removed: 19


In [18]:
removed_df["text"]

246      For their first months of life, breastmilk and...
293      In a frantic hospital hall in northern England...
420      Grain Producers Australia (GPA) has recognised...
687      Guardian Healthcare is a small chain operating...
2652     Our nuclear submarine plan has plenty of risks...
3817     MEDICINE WEB SITE Disclaimer Path to this\nPag...
4125     04 Mar Kyla’s Chronic UTI Story\nReading Time:...
5707     Tyrrell's Vat 47 Chardonnay has had more twist...
8236     IB exam preparation tips and story\nAs I wrap ...
9217     Britain’s primary care system has been the fou...
11121    It is estimated that South Australia will gain...
11848    As we contemplate the real likelihood of Trump...
12373    4500+ Experts Writer\nView AllAmazing Features...
15283    A few years back I was running out of money so...
15971    A crowd of concerned community members packed ...
15984    Health advisor in the UK fined for unlawfully ...
16124    ,Francis Kevin Fogarty was born on 23 October .

In [ ]:
# review removed samples to verify if they are indeed non-au
removed_df["text"][246]

"For their first months of life, breastmilk and/or formula provide babies with all the nutrients and energy they need, but as little ones grow and develop, a more substantial menu is required.\nThings like mashed pumpkin and pureed apple open up a whole new eating experience, giving infants a taste for new foods, a feel for new textures, and vital opportunities to practice chewing and swallowing.\nHowever, as well as powering their bodies and boosting their brains, research indicates that feeding your baby solids earlier can also help them sleep better.\nOver the years, there has been much debate around the age parents should introduce solids to babies.\nAccording to the Department of Health, around six months old is when babies need solid foods added to their liquid diet. This is when they’re likely to show an interest in food, have a bigger appetite, and be able to sit upright with limited support and control their neck and head.\nThis advice is mirrored internationally, with the UK'

From the review, it seems the removed sample is indeed non-au, as it contains "gcse" which is a UK education qualification. This suggests that our filtering based on signal words is effective in identifying non-au content.